In [1]:
!nvidia-smi

Thu Jun  4 14:42:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!ls

Mounted at /content/drive
drive  sample_data


In [3]:
%cd ./drive/MyDrive/ColabNotebooks

/content/drive/MyDrive/ColabNotebooks


In [3]:
!ls

drive  sample_data


# Esecuzione conversione da formato qualsiasi a ppm
### eseguire in caso si voglia cambiare immagine da testare

In [38]:
!python3 convert_to_ppm.py NSB.jpg NSB.ppm

Immagine originale: JPEG, Mode: RGB, Size: (970, 546)
Dopo conversione: Mode: RGB, Size: (970, 546)
✓ Convertito con successo in NSB.ppm
Ora esegui: ./image_to_pgm NSB.ppm output.pgm


## compila solo in caso di:
### modifiche al codice per conversione da ppm a pgm
### testing su immagine differente

In [39]:
!nvcc -O3 -lineinfo image_to_pgm.cu -o img_to_pgm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
image_to_pgm.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%2s", magic);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

image_to_pgm.cu(79): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", width, height);
      ^

image_to_pgm.cu(82): warning #1650-D: result of call is not used
      fscanf(file, "%d", &maxval);
      ^

image_to_pgm.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%2s", magic);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

image_to_pgm.cu(79): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", width, height);
      ^

image_to_pgm.cu(82): warning #1650-D: result of call is not used
      fscanf(file, "%d", &max

# Esecuzione del kernel per conversione da ppm a pgm
### usage:
!./img_to_pgm <input.ppm> <output.ppm>

In [ ]:
!./img_to_pgm NSB.ppm NSB.pgm

=== Conversione Immagine a PGM con CUDA ===
Input: NSB.ppm
Output: NSB.pgm

Dimensioni immagine: 970 x 546
Canali: 3
Grid: (61, 35), Block: (16, 16)

Esecuzione kernel CUDA...
Kernel completato!

Tempo rgbtogray: 114.993149 ms
Scrittura file PGM...
Conversione completata con successo!
File salvato: NSB.pgm


# Compilazione soluzione naive


In [ ]:
!nvcc -O3 -lineinfo DFTcudanaive.cu -o dftnaive

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudanaive.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudanaive.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used


# Esecuzione della soluzione naive 
### Baseline del progetto, realizzata concentrandosi sul solo funzionamento della soluzione e non sull'efficienza
"-f" fa sovrascrittura del file nsight se gia presente su drive

In [ ]:
!ncu --set basic -f -o test_report ./dftnaive 512.pgm

==PROF== Connected to process 40128 (/content/drive/MyDrive/ColabNotebooks/dftnaive)
Immagine caricata di dimensioni H:288 W:512
==PROF== Profiling "dft" - 0: 0%..
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
..50%....100% - 9 passes
Tempo DFT: 248970.937500 ms
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 9 passes
Tempo filtro: 1622.640869 ms
==PROF== Profiling "idft" - 2: 0%....50%....100% - 9 passes
Tempo iDFT: 248021.828125 ms
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 40128
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report.ncu-rep


# Prima ottimizzazione: 
### sono stati usati stratagemmi per ottimizzare l'efficienza dei calcoli matematici
- sincosf
- riduzione della precisione da double a float
- precalcolo delle componenti sin/cos per verticale (costante su u)
- uso di fmaf_rn per operazioni fma non riconosciute come tali dal compilatore e  "-use_fast_math" come argomento del compilatore per riconoscere operazioni adatte al fused multiply add(FMA) e sostituire "sincosf" con un'operazione di basso livello hardware "__sincosf", piu veloce a discapito di una leggera perdita di precisione 


In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt1.cu -o dftopt1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt1.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt1.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      

esecuzione test

In [ ]:
!./dftopt1 Bologna-512.pgm

/bin/bash: line 1: ./dftopt1: Permission denied


### Esecuzione con salvataggio dati per nsight 

In [ ]:
!ncu --set full -f -o test_report_opt1 ./dftopt1 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 30708 (/content/drive/MyDrive/ColabNotebooks/dftopt1)
==PROF== Profiling "dft_opt" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 30708
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt1.ncu-rep


# Seconda ottimizzazione:
### Uso della shared memory
Viene fornitaa shared memory a griglie 16x16 di thread

In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt2.cu -o dftopt2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt2.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt2.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt2.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt2.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2.cu(59): warning #1650-D: result of call is not used
      

esecuzione test

In [ ]:
!./dftopt2 512.pgm

Immagine caricata di dimensioni H:340 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda_opt2-512.pgm


### Esecuzione con salvataggio dati per nsight

In [ ]:
!ncu --set full -f -o test_report_opt2 ./dftopt2 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 31713 (/content/drive/MyDrive/ColabNotebooks/dftopt2)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in o

# Terza ottimizzazione
## Vari test con loop unrolling
l'unrolling viene fatto specificando al compilatore tramite keyword #pragma unroll 'n', dove n è il numero di interazioni in cui spezzare il loop, nel nostro caso operando sul loop interno di dft e idft in seguito alla definizione dei tile come 16x16, faremo unroll per divisore di 16, ovvero 4, 8 e 16


# Test 1: unroll 4



In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt3_4.cu -o dftopt3_4

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt3.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt3.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt3.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt3.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt3.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt3.cu(59): warning #1650-D: result of call is not used
      

test:

In [ ]:
!./dftopt3_4 512.pgm

Immagine caricata di dimensioni H:288 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


esecuzione con raccolta report

In [ ]:
!ncu --set full -f -o test_report_opt3_4 ./dftopt3_4 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 33604 (/content/drive/MyDrive/ColabNotebooks/dftopt3)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_c

# Test 2: unroll 8



In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt3_8.cu -o dftopt3_8

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4.cu(59): warning #1650-D: result of call is not used
      

test:

In [ ]:
!./dftopt3_8 512.pgm

Immagine caricata di dimensioni H:288 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


esecuzione con raccolta report

In [ ]:
!ncu --set full -f -o test_report_opt3_8 ./dftopt3_8 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 37352 (/content/drive/MyDrive/ColabNotebooks/dftopt4)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_c

# Test 3: unroll 16



In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt3_16.cu -o dftopt3_16

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt5.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt5.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt5.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt5.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(58): warning #1650-D: result of call is not used
      

test:

In [ ]:
!./dftopt3_16 512.pgm

esecuzione con raccolta report

In [ ]:
!ncu --set full -f -o test_report_opt3_16 ./dftopt3_16 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 38307 (/content/drive/MyDrive/ColabNotebooks/dftopt5)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_c

### Si è notato un incremento delle performance solo nel momento in cui è stato fatto loop unrolling di 16 istruzione, dato che va a arimuovere completamente l'operazione condizionale del ciclo

## Correzione, i loop unroll hanno evidenziato un incremento delle prestazioni molto elevato con shared memory, testiamo idf shared
# unroll 4


In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt4_4.cu -o dftopt4_4


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4_4.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_4.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4_4.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4_4.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4_4.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_4.cu(59): warning #1650-D: result of call is not

In [ ]:
!./dftopt4_4 512.pgm

In [ ]:
!ncu --set full -f -o test_report_opt4_4 ./dftopt4_4 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 5236 (/content/drive/MyDrive/ColabNotebooks/dftopt4_4)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in 

# unroll 8

In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt4_8.cu -o dftopt4_8


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4_8.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_8.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4_8.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4_8.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4_8.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_8.cu(59): warning #1650-D: result of call is not

In [ ]:
!./dftopt4_8 512.pgm

In [ ]:
!ncu --set full -f -o test_report_opt4_8 ./dftopt4_8 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 6346 (/content/drive/MyDrive/ColabNotebooks/dftopt4_8)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_shared" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 6346
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt4_8.ncu-rep


# unroll 16

In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt4_16.cu -o dftopt4_16


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4_16.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_16.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4_16.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4_16.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4_16.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4_16.cu(58): warning #1650-D: result of call 

In [ ]:
!./dftopt4_16 512.pgm

In [ ]:
!ncu --set full -f -o test_report_opt4_16 ./dftopt4_16 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 9352 (/content/drive/MyDrive/ColabNotebooks/dftopt4_16)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_shared" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 9352
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt4_16.ncu-rep


# Quarta ottimizzazione
### Cache L1 configuration

## _ _restrict__ sui puntatori nella firma dei kernel:
- dice al compilatore che in e out non si sovrappongono in memoria
## Vector store con float2 per la DFT:
- La scrittura di MyComplex (due float) può essere fatta come store vettoriale se la struct è allineata a 8 byte — una singola transazione invece di due, (IMPORTANTE specificare la keyword align(8) davanti alla struct mycomplex per migliorare la coalescenza)

In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt5.cu -o dftopt5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt5.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt5.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt5.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt5.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(59): warning #1650-D: result of call is not used
      

testing

In [ ]:
!./dftopt5 512.pgm

### profiling

In [ ]:
!ncu --set full -f -o test_report_opt5 ./dftopt5 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 28526 (/content/drive/MyDrive/ColabNotebooks/dftopt5)
==PROF== Profiling "dft_opt_L1" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_L1" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 28526
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt5.ncu-rep


# TESTING EXTRA 4SHAREDMEMORY

In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt2-5.cu -o dftopt2-5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt2-5.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2-5.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt2-5.cu(60): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt2-5.cu(64): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt2-5.cu(53): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2-5.cu(59): warning #1650-D: result of call is not

In [ ]:
!ncu --set full -f -o test_report_opt2-5 ./dftopt2-5 512.pgm

Immagine caricata di dimensioni H:288 W:512
==PROF== Connected to process 36207 (/content/drive/MyDrive/ColabNotebooks/dftopt2-5)
==PROF== Profiling "dft_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output

# ESTENSIONE: EDGE DETECTION SU VIDEO
### implementazione naive su video mp4 usando script per estrazione frame e conversione in pgm, poi script per ricomposizione video

In [ ]:
# Installa opencv C++ headers e librerie
!apt-get install -y libopencv-dev

# Verifica che pkg-config la trovi ora
!pkg-config --cflags --libs opencv2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopencv-dev is already the newest version (4.5.4+dfsg-9ubuntu4+jammy1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
Package opencv2 was not found in the pkg-config search path.
Perhaps you should add the directory containing `opencv2.pc'
to the PKG_CONFIG_PATH environment variable
Package 'opencv2', required by 'virtual:world', not found


In [ ]:
# Estrai i frame dal video e ridimensionali a 960x540
!ffmpeg -i ./dashcamsd.mp4 -vf scale=960:540 frames/frame_%04d.pgm


In [ ]:

# 3. Compila il tuo codice CUDA invariato
#!nvcc -O2 -use_fast_math -lineinfo -arch=sm_75 -o dftopt5 DFTcudaOpt5.cu

# 4. Processa i frame
!for f in $(ls frames/frame_*.pgm | sort); do ./dftopt5 "$f" && mv output_cuda-512.pgm "output/$(basename $f)"; done


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [40]:

# 5. Riassembla
!ffmpeg -framerate 25 -i output/frame_%04d.pgm -c:v libx264 output_edge.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# loop c++ con pinned memory

In [42]:
!nvcc -O2 -use_fast_math -arch=sm_75 -lineinfo cudaVideo.cu -o cudaVideo

cudaVideo.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

cudaVideo.cu(63): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

cudaVideo.cu(64): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

cudaVideo.cu(68): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

cudaVideo.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

cudaVideo.cu(63): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

cudaVideo.cu(64): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

cudaVideo.cu(68): warning #16

### non credo abbia senso fare questo perchè sono i soliti kernel, solo ripetuti per ogni frame, quindi ci mette un botto

In [43]:
!ncu --set full -f -o test_report_video1 ./cudaVideo ./frames ./

==PROF== Connected to process 4984 (/content/drive/MyDrive/ColabNotebooks/cudaVideo)
==PROF== Profiling "dft_opt_L1" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft_opt_L1" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "dft_opt_L1" - 3: 0%.==PROF== Trying to shutdown target application
 - 1 pass
==ERROR== Failed t

# OPT 1 - Doppio Stream CUDA, overlap tra transfer e compute

### vedere i commenti sul codice che spiegano tanto

In [7]:
!nvcc -O2 -use_fast_math -arch=sm_75 -lineinfo optVideo1.cu -o optVideo1

optVideo1.cu(54): warning #1650-D: result of call is not used
      (void)fscanf(file, "%s", format);
            ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

optVideo1.cu(54): warning #1650-D: result of call is not used
      (void)fscanf(file, "%s", format);
      ^

optVideo1.cu(58): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d %d", &img.width, &img.height);
            ^

optVideo1.cu(58): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d %d", &img.width, &img.height);
      ^

optVideo1.cu(59): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d", &img.max_value);
            ^

optVideo1.cu(59): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d", &img.max_value);
      ^

optVideo1.cu(63): warning #1650-D: result of call is not used
      (void)fread(img.data, 1, img.width * img.height, file);
            ^

optVideo1.cu(63): warning #1650-D: result 

In [20]:
!nvcc -O2 -use_fast_math -arch=sm_75 -lineinfo ./drive/MyDrive/ColabNotebooks/optVideo1.cu -o optVideo1

./drive/MyDrive/ColabNotebooks/optVideo1.cu(54): warning #1650-D: result of call is not used
      (void)fscanf(file, "%s", format);
            ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

./drive/MyDrive/ColabNotebooks/optVideo1.cu(54): warning #1650-D: result of call is not used
      (void)fscanf(file, "%s", format);
      ^

./drive/MyDrive/ColabNotebooks/optVideo1.cu(58): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d %d", &img.width, &img.height);
            ^

./drive/MyDrive/ColabNotebooks/optVideo1.cu(58): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d %d", &img.width, &img.height);
      ^

./drive/MyDrive/ColabNotebooks/optVideo1.cu(59): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d", &img.max_value);
            ^

./drive/MyDrive/ColabNotebooks/optVideo1.cu(59): warning #1650-D: result of call is not used
      (void)fscanf(file, "%d", &img.max_value);
    

In [21]:
!chmod +x ./optVideo1

In [9]:
!ls

drive						     optVideo1
nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb    sample_data
nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb.1


In [28]:
%cd ..

/content


In [4]:
%cp  ./optVideo1 ../../../optVideo1

In [ ]:
!ncu --set full -f -o test_report_videoopt1 ./optVideo1 ./frames ./outputOpt1

==PROF== Connected to process 14702 (/content/drive/MyDrive/ColabNotebooks/optVideo1)
Device: Tesla T4 (sm_75)
Shared mem/SM: 64 KB
Frame trovati: 818
Dimensioni: 960x540
==PROF== Profiling "dft_opt_L1" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "idft_filtered" - 1: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "motion_detect" - 2: 0%....50%....100% - 31 passes
Frame 1/818 elaborat

In [5]:
# 1. Scarica una versione stabile di Nsight Systems per Ubuntu 22.04 (il SO di base di Colab)
!wget -q https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64/nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb

# 2. Aggiorna la lista dei pacchetti
!apt-get update -q

# 3. Installa il pacchetto scaricato
!apt-get install -yq ./nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb

# 4. Risolve eventuali dipendenze mancanti
!apt-get --fix-broken install -yq

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lis

In [17]:
import os
import stat

path_out = '/content/drive/MyDrive/ColabNotebooks/output_edges'

# Controlla permessi
st = os.stat(path_out)
print(f"Permessi: {oct(st.st_mode)}")
print(f"Scrivibile: {os.access(path_out, os.W_OK)}")

# Tentativo diretto di scrittura da Python
try:
    with open(f"{path_out}/test.txt", 'w') as f:
        f.write("test")
    print("Scrittura OK")
    os.remove(f"{path_out}/test.txt")
except Exception as e:
    print(f"Scrittura fallita: {e}")

Permessi: 0o40700
Scrivibile: True
Scrittura OK


In [23]:
!nsys profile --trace=cuda,osrt --stats=true --force-overwrite=true --output=statsOpt1  ./optVideo1 ./drive/MyDrive/ColabNotebooks/frames ./drive/MyDrive/ColabNotebooks/output_edges

Output: ./drive/MyDrive/ColabNotebooks/output_edges
Device: Tesla T4 (sm_75)
Frame trovati: 818
Dimensioni: 960x540
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0000.pgm
Frame 1/818 elaborato
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0001.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0002.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0003.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0004.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0005.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0006.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0007.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0008.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0009.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0010.pgm
Frame 11/818 elaborato
Scrivo: ./drive/MyDrive/ColabNotebooks/output_edges/edge_0011.pgm
Scrivo: ./drive/MyDrive/ColabNotebooks/output_e

In [24]:
!cp statsOpt1.nsys-rep /content/drive/MyDrive/ColabNotebooks/statsOpt1.nsys-rep

In [ ]:

# 5. Riassembla
!ffmpeg -framerate 25 -i ./drive/MyDrive/ColabNotebooks/output_edges/edge_%04d.pgm -c:v libx264 ./drive/MyDrive/ColabNotebooks/output_edgeopt1.1.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [10]:
%cd ../../..

/content


In [6]:
!ls

drive
nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb
nsight-systems-2023.2.3_2023.2.3.1001-1_amd64.deb.1
sample_data


In [73]:
# Forza la creazione di tutta la catena di cartelle tramite Bash (-p)
!mkdir -p /content/drive/MyDrive/ColabNotebooks/output_edges
